In [ ]:
import json, glob, os
import pandas as pd
import matplotlib.pyplot as plt

def load_results(prefix):
    records = []
    for path in glob.glob(f"results/{prefix}_*.json"):
        with open(path) as f:
            records.append(json.load(f))
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records)
    # Keep only the latest run per unique (adapter, operation, clients) combination
    df = df.sort_values("timestamp").groupby(["adapter", "operation", "clients"], as_index=False).last()
    return df


In [ ]:
# Chart 1: Insert throughput (records/sec) per adapter
df = load_results("insert")
if df.empty:
    print("No insert results found. Run benchmark_insert.py first.")
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(df["adapter"], df["records_per_sec"], color="steelblue")
    ax.set_xlabel("Adapter")
    ax.set_ylabel("Records / sec")
    ax.set_title("Insert Throughput")
    plt.tight_layout()
    plt.show()
    display(df[["adapter", "records_per_sec"]].sort_values("records_per_sec", ascending=False).reset_index(drop=True))

**How to read this chart:** The adapter with the **tallest bar** has the highest insert throughput (records/sec) — that is the best choice for bulk ingestion workloads. If bars are close in height, the difference is negligible in practice.

In [ ]:
# Chart 2: Query latency ANN (p50/p95/p99)
df = load_results("query")
df_ann = df[df["operation"] == "query_ann"]
if df_ann.empty:
    print("No ANN query results found. Run benchmark_query.py first.")
else:
    x = range(len(df_ann))
    width = 0.25
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar([i - width for i in x], df_ann["p50_ms"], width, label="p50")
    ax.bar(list(x), df_ann["p95_ms"], width, label="p95")
    ax.bar([i + width for i in x], df_ann["p99_ms"], width, label="p99")
    ax.set_xticks(list(x))
    ax.set_xticklabels(df_ann["adapter"].tolist())
    ax.set_ylabel("Latency (ms)")
    ax.set_title("ANN Query Latency")
    ax.legend()
    plt.tight_layout()
    plt.show()
    display(df_ann[["adapter", "p50_ms", "p95_ms", "p99_ms", "qps"]].sort_values("p99_ms").reset_index(drop=True))

**How to read this chart:** **Lower bars are better** (less latency). Prioritize p99 if tail latency matters (e.g. interactive search); p50 reflects the typical query experience. The adapter with the shortest p99 bar gives the most consistent ANN performance.

In [ ]:
# Chart 3: Filtered query latency (p50/p95/p99)
df = load_results("query")
df_filt = df[df["operation"] == "query_filtered"]
if df_filt.empty:
    print("No filtered query results found. Run benchmark_query.py first.")
else:
    x = range(len(df_filt))
    width = 0.25
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar([i - width for i in x], df_filt["p50_ms"], width, label="p50")
    ax.bar(list(x), df_filt["p95_ms"], width, label="p95")
    ax.bar([i + width for i in x], df_filt["p99_ms"], width, label="p99")
    ax.set_xticks(list(x))
    ax.set_xticklabels(df_filt["adapter"].tolist())
    ax.set_ylabel("Latency (ms)")
    ax.set_title("Filtered Query Latency")
    ax.legend()
    plt.tight_layout()
    plt.show()
    display(df_filt[["adapter", "p50_ms", "p95_ms", "p99_ms", "qps"]].sort_values("p99_ms").reset_index(drop=True))

**How to read this chart:** Same as Chart 2 — **lower bars are better**. If filtered query latency is much higher than ANN latency for an adapter, it handles metadata filtering inefficiently. Prefer the adapter whose filtered p99 stays closest to its ANN p99.

In [ ]:
# Chart 4: Concurrency degradation — p99 latency vs client count
df = load_results("concurrency")
if df.empty:
    print("No concurrency results found. Run benchmark_concurrency.py first.")
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    for adapter, grp in df.groupby("adapter"):
        grp_sorted = grp.sort_values("clients")
        ax.plot(grp_sorted["clients"], grp_sorted["p99_ms"], marker="o", label=adapter)
    ax.set_xlabel("Concurrent Clients")
    ax.set_ylabel("p99 Latency (ms)")
    ax.set_title("Concurrency Degradation (p99)")
    ax.legend()
    plt.tight_layout()
    plt.show()
    display(df[["adapter", "clients", "qps", "p50_ms", "p99_ms"]].sort_values(["clients", "p99_ms"]).reset_index(drop=True))

**How to read this chart:** A **flatter line** means the adapter degrades less under load — better for concurrent workloads. A steep upward curve means latency spikes as clients increase. Look for the adapter whose p99 line rises the least from left to right.

In [ ]:
# Chart 5: Query QPS (throughput) — ANN and filtered side by side
df = load_results("query")
if df.empty:
    print("No query results found.")
else:
    df_ann  = df[df["operation"] == "query_ann"].set_index("adapter")
    df_filt = df[df["operation"] == "query_filtered"].set_index("adapter")
    adapters = df_ann.index.tolist()
    x = range(len(adapters))
    width = 0.35
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar([i - width/2 for i in x], df_ann.loc[adapters, "qps"],  width, label="ANN",      color="steelblue")
    ax.bar([i + width/2 for i in x], df_filt.loc[adapters, "qps"], width, label="Filtered",  color="coral")
    ax.set_xticks(list(x))
    ax.set_xticklabels(adapters)
    ax.set_ylabel("Queries / sec")
    ax.set_title("Query Throughput (QPS)")
    ax.legend()
    plt.tight_layout()
    plt.show()
    qps_summary = pd.DataFrame({
        "adapter": adapters,
        "ann_qps": df_ann.loc[adapters, "qps"].values,
        "filtered_qps": df_filt.loc[adapters, "qps"].values,
    })
    qps_summary["filtered_vs_ann_%"] = (qps_summary["filtered_qps"] / qps_summary["ann_qps"] * 100).round(1)
    display(qps_summary.sort_values("ann_qps", ascending=False).reset_index(drop=True))

**How to read this chart:** **Taller bars are better** (more queries per second). Compare ANN vs. Filtered bars side by side for each adapter — a large drop from ANN to Filtered indicates the adapter struggles with metadata filtering at scale. The overall best adapter maximizes both bars.

In [ ]:
# Scorecard: normalize each metric to 0–1 and rank adapters
# Higher score = better. Latency metrics are inverted (lower latency → higher score).
import numpy as np

def norm_higher(s): return (s - s.min()) / (s.max() - s.min() + 1e-9)
def norm_lower(s):  return 1 - norm_higher(s)

di = load_results("insert").set_index("adapter")
dq = load_results("query")
dqa = dq[dq["operation"] == "query_ann"].set_index("adapter")
dqf = dq[dq["operation"] == "query_filtered"].set_index("adapter")
dc  = load_results("concurrency")
dc_max = dc[dc["clients"] == dc["clients"].max()].set_index("adapter")

adapters = sorted(set(di.index) & set(dqa.index) & set(dqf.index) & set(dc_max.index))

scorecard = pd.DataFrame(index=adapters)
scorecard["insert_rps"]       = norm_higher(di.loc[adapters, "records_per_sec"])
scorecard["ann_p50"]          = norm_lower(dqa.loc[adapters, "p50_ms"])
scorecard["ann_p99"]          = norm_lower(dqa.loc[adapters, "p99_ms"])
scorecard["filtered_p50"]     = norm_lower(dqf.loc[adapters, "p50_ms"])
scorecard["filtered_p99"]     = norm_lower(dqf.loc[adapters, "p99_ms"])
scorecard["concurrency_qps"]  = norm_higher(dc_max.loc[adapters, "qps"])
scorecard["concurrency_p99"]  = norm_lower(dc_max.loc[adapters, "p99_ms"])
scorecard["overall"]          = scorecard.mean(axis=1)
scorecard = scorecard.sort_values("overall", ascending=False)

print("=== Scorecard (0 = worst, 1 = best) ===\n")
print(scorecard.round(3).to_string())
print(f"\nOverall winner: {scorecard.index[0].upper()}")


In [ ]:
# Chart 6: Radar chart — all dimensions at once
labels = ["insert_rps", "ann_p50", "ann_p99", "filtered_p50", "filtered_p99", "concurrency_qps", "concurrency_p99"]
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = plt.cm.tab10.colors

for i, adapter in enumerate(scorecard.index):
    values = scorecard.loc[adapter, labels].tolist()
    values += values[:1]
    ax.plot(angles, values, "o-", linewidth=2, label=adapter, color=colors[i % len(colors)])
    ax.fill(angles, values, alpha=0.08, color=colors[i % len(colors)])

ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.50", "0.75", "1.0"], fontsize=8)
ax.set_title("Overall Comparison (larger = better)", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()


In [ ]:
# Summary: per-use-case recommendation
di_raw  = load_results("insert").set_index("adapter")
dqa_raw = load_results("query")[lambda d: d["operation"] == "query_ann"].set_index("adapter")
dc_raw  = load_results("concurrency")
dc_max_raw = dc_raw[dc_raw["clients"] == dc_raw["clients"].max()].set_index("adapter")

best_insert      = di_raw["records_per_sec"].idxmax()
best_ann_latency = dqa_raw["p50_ms"].idxmin()
best_ann_qps     = dqa_raw["qps"].idxmax()
best_concurrent  = dc_max_raw["qps"].idxmax()
best_overall     = scorecard.index[0]

print("========================================")
print("        USE-CASE RECOMMENDATIONS        ")
print("========================================")
print(f"  Bulk ingestion speed     →  {best_insert}")
print(f"  Lowest ANN latency (p50) →  {best_ann_latency}")
print(f"  Highest single-client QPS→  {best_ann_qps}")
print(f"  Best under concurrency   →  {best_concurrent}")
print(f"  Best overall (balanced)  →  {best_overall}")
print("========================================")
print()

notes = {
    "sqlite_vec": "Embedded, zero-infra, great for single-process or edge workloads.",
    "pgvector":   "Best if you already run Postgres; transactional + vector in one DB.",
    "weaviate":   "Purpose-built vector DB; rich filtering and schema support.",
    "pinecone":   "Fully managed cloud; no ops overhead but adds network latency.",
}
for adapter in scorecard.index:
    note = notes.get(adapter, "")
    score = scorecard.loc[adapter, "overall"]
    print(f"  {adapter:<14} (score {score:.2f})  {note}")


# Vector DB Benchmark — Glossary

## Latency Percentiles

| Term | Full Name | What it means |
|------|-----------|----------------|
| **p50** | 50th percentile (median) | Half of all queries completed faster than this. Represents the *typical* user experience. |
| **p95** | 95th percentile | 95% of queries were faster than this. Starts to reveal occasional slow outliers. |
| **p99** | 99th percentile (tail latency) | 99% of queries were faster than this. The worst-case latency most users will ever see. Use this to judge consistency under load. |

> **Rule of thumb:** p50 shows average performance; p99 shows whether the system has latency spikes. A good database has a small gap between p50 and p99.

---

## Throughput

| Term | Full Name | What it means |
|------|-----------|----------------|
| **QPS** | Queries Per Second | How many search queries the database can handle per second. Higher = better. |
| **records/sec** (RPS) | Records Per Second | How many records can be inserted per second during bulk ingestion. Higher = better. |

---

## Query Types

| Term | Full Name | What it means |
|------|-----------|----------------|
| **ANN** | Approximate Nearest Neighbor | The core vector search operation — finds the *k* most similar vectors to a query vector. "Approximate" means it trades a small accuracy loss for much faster speed compared to exact search. |
| **Filtered query** | Metadata-filtered ANN search | ANN search combined with a metadata condition (e.g. `category = "news"`). Harder than plain ANN — databases must narrow the vector space while respecting the filter. |

---

## Load & Infrastructure

| Term | Full Name | What it means |
|------|-----------|----------------|
| **Concurrent clients** | Simultaneous connections | Number of parallel threads/processes sending queries at the same time. Higher concurrency stresses the database and reveals how latency degrades under real-world multi-user load. |
| **Concurrency degradation** | Latency growth under load | How much p99 latency increases as concurrent clients go up. A flat curve = scales well; a steep curve = bottlenecks under load. |
| **Adapter** | Database driver/connector | The wrapper used to talk to each vector database (pgvector, sqlite_vec, weaviate, pinecone). Each adapter implements the same benchmark interface so results are comparable. |

---

## Scoring

| Term | What it means |
|------|----------------|
| **Normalized score (0–1)** | Each metric is scaled so 1.0 = best adapter on that metric, 0.0 = worst. Allows apples-to-apples comparison across metrics with different units. |
| **Overall score** | Simple average of all normalized metric scores. Higher = better balanced across all dimensions. |

## Before vs After Comparison

Set `BEFORE_LABEL` and `AFTER_LABEL` below, then run the cells to compare two benchmark runs.

In [ ]:
import numpy as np

BEFORE_LABEL = "baseline"
AFTER_LABEL  = "optimized"

def load_comparison(operation):
    """Load before/after DataFrames for a given operation, keyed by adapter."""
    import glob as _glob
    all_records = []
    for path in _glob.glob("results/*.json"):
        with open(path) as f:
            r = json.load(f)
        r.setdefault("label", "baseline")
        all_records.append(r)

    if not all_records:
        return pd.DataFrame(), pd.DataFrame()

    df = pd.DataFrame(all_records)
    df = df[df["operation"] == operation]

    before = (df[df["label"] == BEFORE_LABEL]
              .sort_values("timestamp")
              .groupby("adapter").last())
    after  = (df[df["label"] == AFTER_LABEL]
              .sort_values("timestamp")
              .groupby("adapter").last())
    return before, after


def get_throughput_col(df, operation):
    """Return records_per_sec for insert, qps for all others."""
    if operation == "insert" and "records_per_sec" in df.columns:
        return df["records_per_sec"]
    return df["qps"]

In [ ]:
# Grouped bar chart: before/after p50 and p99 per adapter, per operation
for op in ["query_ann", "query_filtered", "insert", "concurrency"]:
    before, after = load_comparison(op)
    if before.empty or after.empty:
        print(f"No {BEFORE_LABEL}/{AFTER_LABEL} comparison data for {op} — run benchmarks with --label first.")
        continue

    adapters = sorted(set(before.index) & set(after.index))
    if not adapters:
        continue

    x = np.arange(len(adapters))
    width = 0.2

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - 1.5 * width, before.loc[adapters, "p50_ms"], width,
           label=f"{BEFORE_LABEL} p50", color="steelblue", alpha=0.55)
    ax.bar(x - 0.5 * width, after.loc[adapters, "p50_ms"],  width,
           label=f"{AFTER_LABEL} p50",  color="steelblue")
    ax.bar(x + 0.5 * width, before.loc[adapters, "p99_ms"], width,
           label=f"{BEFORE_LABEL} p99", color="coral", alpha=0.55)
    ax.bar(x + 1.5 * width, after.loc[adapters, "p99_ms"],  width,
           label=f"{AFTER_LABEL} p99",  color="coral")

    ax.set_xticks(x)
    ax.set_xticklabels(adapters)
    ax.set_ylabel("Latency (ms)")
    ax.set_title(f"{op} — {BEFORE_LABEL} vs {AFTER_LABEL}  (p50 / p99, lower is better)")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# % improvement table — green = improvement, red = regression
def color_delta(val, lower_is_better=True):
    if pd.isna(val):
        return ""
    if lower_is_better:
        if val < -1:
            return "background-color: #c6efce; color: #276221"
        if val > 1:
            return "background-color: #ffc7ce; color: #9c0006"
    else:
        if val > 1:
            return "background-color: #c6efce; color: #276221"
        if val < -1:
            return "background-color: #ffc7ce; color: #9c0006"
    return ""


for op in ["query_ann", "query_filtered", "insert", "concurrency"]:
    before, after = load_comparison(op)
    if before.empty or after.empty:
        continue

    adapters = sorted(set(before.index) & set(after.index))
    if not adapters:
        continue

    thr_before = get_throughput_col(before.loc[adapters], op)
    thr_after  = get_throughput_col(after.loc[adapters],  op)

    def pct(bv, av):
        return round((av - bv) / bv * 100, 1) if bv else 0.0

    rows = []
    for adapter in adapters:
        b, a = before.loc[adapter], after.loc[adapter]
        rows.append({
            "adapter":    adapter,
            "p50 before": round(b["p50_ms"], 2),
            "p50 after":  round(a["p50_ms"], 2),
            "p50 Δ%":     pct(b["p50_ms"], a["p50_ms"]),
            "p99 before": round(b["p99_ms"], 2),
            "p99 after":  round(a["p99_ms"], 2),
            "p99 Δ%":     pct(b["p99_ms"], a["p99_ms"]),
            "thru before": round(thr_before.loc[adapter], 2),
            "thru after":  round(thr_after.loc[adapter],  2),
            "thru Δ%":    pct(thr_before.loc[adapter], thr_after.loc[adapter]),
        })

    tbl = pd.DataFrame(rows).set_index("adapter")
    print(f"\n{'='*60}\n  {op.upper()}  ({BEFORE_LABEL} → {AFTER_LABEL})\n{'='*60}")
    styled = (tbl.style
                 .map(lambda v: color_delta(v, lower_is_better=True),  subset=["p50 Δ%", "p99 Δ%"])
                 .map(lambda v: color_delta(v, lower_is_better=False), subset=["thru Δ%"])
                 .format("{:.2f}", subset=["p50 before", "p50 after", "p99 before", "p99 after",
                                           "thru before", "thru after"])
                 .format("{:+.1f}%", subset=["p50 Δ%", "p99 Δ%", "thru Δ%"]))
    display(styled)

In [ ]:
# Scorecard: baseline vs optimized — per-metric and overall rankings for both runs
def load_results_labeled(prefix, label):
    """Load latest result per (adapter, operation, clients) for a specific label."""
    import glob as _glob
    records = []
    for path in _glob.glob(f"results/{prefix}_*.json"):
        with open(path) as f:
            r = json.load(f)
        r.setdefault("label", "baseline")
        if r.get("label") == label:
            records.append(r)
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records)
    df = df.sort_values("timestamp").groupby(["adapter", "operation", "clients"], as_index=False).last()
    return df


def compute_scorecard(label):
    di  = load_results_labeled("insert",      label).set_index("adapter") if not load_results_labeled("insert",      label).empty else pd.DataFrame()
    dq  = load_results_labeled("query",       label)
    dc  = load_results_labeled("concurrency", label)

    if di.empty or dq.empty or dc.empty:
        return pd.DataFrame(), label

    dqa     = dq[dq["operation"] == "query_ann"].set_index("adapter")
    dqf     = dq[dq["operation"] == "query_filtered"].set_index("adapter")
    dc_max  = dc[dc["clients"] == dc["clients"].max()].set_index("adapter")

    adapters = sorted(set(di.index) & set(dqa.index) & set(dqf.index) & set(dc_max.index))
    if not adapters:
        return pd.DataFrame(), label

    def norm_higher(s): return (s - s.min()) / (s.max() - s.min() + 1e-9)
    def norm_lower(s):  return 1 - norm_higher(s)

    sc = pd.DataFrame(index=adapters)
    sc["insert_rps"]      = norm_higher(di.loc[adapters, "records_per_sec"])
    sc["ann_p50"]         = norm_lower(dqa.loc[adapters, "p50_ms"])
    sc["ann_p99"]         = norm_lower(dqa.loc[adapters, "p99_ms"])
    sc["filtered_p50"]    = norm_lower(dqf.loc[adapters, "p50_ms"])
    sc["filtered_p99"]    = norm_lower(dqf.loc[adapters, "p99_ms"])
    sc["concurrency_qps"] = norm_higher(dc_max.loc[adapters, "qps"])
    sc["concurrency_p99"] = norm_lower(dc_max.loc[adapters, "p99_ms"])
    sc["overall"]         = sc.mean(axis=1)
    return sc.sort_values("overall", ascending=False), label


for label in [BEFORE_LABEL, AFTER_LABEL]:
    sc, lbl = compute_scorecard(label)
    if sc.empty:
        print(f"No data for label '{lbl}' — run benchmarks with --label {lbl} first.\n")
        continue

    print(f"\n{'='*55}")
    print(f"  SCORECARD — {lbl.upper()}  (0 = worst, 1 = best)")
    print(f"{'='*55}")
    display(sc.round(3).style.highlight_max(color="lightgreen", axis=0).format("{:.3f}"))

    notes = {
        "sqlite_vec": "Embedded, zero-infra, great for single-process or edge.",
        "pgvector":   "Transactional + vector in one DB if you already run Postgres.",
        "weaviate":   "Purpose-built vector DB; rich filtering and schema support.",
        "pinecone":   "Fully managed cloud; no ops overhead but adds network latency.",
    }
    print(f"\n  Overall winner ({lbl}): {sc.index[0].upper()}\n")
    for adapter in sc.index:
        note  = notes.get(adapter, "")
        score = sc.loc[adapter, "overall"]
        best_metric = sc.loc[adapter, sc.columns[:-1]].idxmax()
        print(f"  {adapter:<14} overall={score:.3f}  best at: {best_metric:<20}  {note}")
    print()